# 02 — Feature Engineering

Walks through the four feature modules that transform raw list-format signals into
per-applicant scalar features used by the XGBoost model.

Modules:
- `upi_features.py` — 7 UPI features (6-month window)
- `utility_features.py` — 4 utility bill features
- `mobile_features.py` — 4 mobile recharge features
- `gst_features.py` — 3 GST features (MSME only)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.features.build_features import expand_list_columns
from src.features import upi_features, utility_features, mobile_features, gst_features
from src.feature_engine import build_feature_matrix

PROFILE_PATH = '../data/synthetic/profiles.parquet'
df_raw = pd.read_parquet(PROFILE_PATH)
print(f'Raw profiles: {len(df_raw):,} rows x {len(df_raw.columns)} cols')

## 1. Expand List Columns

Raw profiles store 12-month signals as Python lists per cell.  
`expand_list_columns()` flattens them to `upi_count_m1 … upi_count_m12`.

In [ ]:
df_expanded = expand_list_columns(df_raw.head(5))
upi_cols = [c for c in df_expanded.columns if c.startswith('upi_count')]
print(f'UPI count columns: {upi_cols}')
df_expanded[['applicant_id'] + upi_cols]

## 2. UPI Feature Transform

In [ ]:
# Apply to a small sample for inspection
sample = expand_list_columns(df_raw.head(500).copy())
sample_upi = upi_features.transform(sample)

upi_feature_cols = [
    'upi_txn_count_6m', 'upi_consistency_score', 'upi_merchant_diversity',
    'upi_failed_rate', 'upi_avg_txn_value', 'upi_night_txn_share', 'upi_income_regularity'
]
sample_upi[upi_feature_cols].describe().round(3)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for i, col in enumerate(upi_feature_cols):
    axes[i].hist(sample_upi[col].dropna(), bins=30, color='#0066FF', edgecolor='#0A0A0A', linewidth=0.6)
    axes[i].set_title(col.replace('upi_', '').replace('_', ' '), fontsize=9, fontweight='bold')
axes[-1].set_visible(False)
plt.suptitle('UPI Feature Distributions (500 sample)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Full Feature Matrix via `build_feature_matrix()`

In [ ]:
# This runs the entire pipeline: expand → UPI → utility → mobile → GST
PROCESSED_PATH = '../data/processed/features.parquet'

if not os.path.exists(PROCESSED_PATH):
    print('Running full feature engineering pipeline...')
    import subprocess
    result = subprocess.run(['python', '../src/feature_engine.py'], capture_output=True, text=True)
    print(result.stdout[-2000:] if result.stdout else result.stderr[-2000:])

df_features = pd.read_parquet(PROCESSED_PATH)
print(f'Feature matrix: {df_features.shape}')
df_features.head(3)

## 4. Feature Correlations with Default Label

In [ ]:
ML_FEATURES = [
    'upi_txn_count_6m', 'upi_consistency_score', 'upi_merchant_diversity',
    'upi_failed_rate', 'upi_avg_txn_value', 'upi_night_txn_share', 'upi_income_regularity',
    'utility_streak_length', 'utility_days_before_due_avg',
    'utility_lapse_count_12m', 'utility_reinstatement_count',
    'mobile_plan_tier', 'mobile_recharge_streak', 'mobile_plan_trend', 'mobile_lapse_count',
    'gst_filing_regularity', 'gst_turnover_trend', 'gst_penalty_count',
]

available = [c for c in ML_FEATURES if c in df_features.columns]
corrs = df_features[available + ['default_label']].corr()['default_label'].drop('default_label').sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
colors = ['#D50000' if v > 0 else '#0066FF' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors, edgecolor='#0A0A0A', linewidth=0.8)
ax.axvline(0, color='#0A0A0A', linewidth=1.2)
ax.set_title('Pearson Correlation with Default Label', fontweight='bold', fontsize=12)
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.savefig('../models/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Value Ranges — Sanity Check

In [ ]:
df_features[available].describe().round(4)